# Lecture 15 - Interpolation

We frequently encounter situations where we only have data at a
discrete number of points, or a function is very to expensive to
compute _on demand_, so we tabulate it at several points.

[Interpolation](https://en.wikipedia.org/wiki/Interpolation) fills in the gaps in our data by making an assumption about the
behavior of the underlying functional form of the data.

Many types of interpolation exists:
* Some ensure that no new extrema a introduced.
* Some ensure a physical constraint (like thermodynamic consistency) is satisfied.
* Some ensure smoothness of the fit.
* Some ensure the quantity being interpolated is conserved.

Generally speaking, you need to balance the number of points used in
constructing an interpolant (which _can_ increase accuracy) against
pathologies (like oscillations).

### Accepting and submitting

To accept this assignment on Classroom 50 with your GitHub profile enrolled in the class, click on the link \
https://classroom50.org/PsuAstro410/410astro26/assignments/lecture-15/accept \
and follow the steps the webpage prompts.

To submit this assignment, follow the steps in "First time Classroom 50 setup" and "Accepting and Submitting Assignments" on the course webpage:\
https://psuastro410.github.io/tips/github/

### Resources and Acknowledgements

We have used material from the following sources:\
https://zingale.github.io/computational_astrophysics/interpolation/interpolation.html \
https://zingale.github.io/computational_astrophysics/interpolation/lagrange-interpolation.html \
https://zingale.github.io/computational_astrophysics/interpolation/application-interpolating-reaction-rates.html \
https://zingale.github.io/computational_astrophysics/homework/homework6.html \
https://zingale.github.io/computational_astrophysics/homework/homework6-solutions.html

# Project Outline, due Oct. 27

Outlines will map out projects in some more depth, and be approximately two pages in length of text and equations, submitted as a pdf using Overleaf. Your outline should include

* The scientific problem your work aims to address
* The analysis and/or calculations your work will perform
* Primary algorithms implemented, including ones discussed in-class to complete exam
* Inputs and outputs
* At least two possible tests

I will have given you comments by now on suggestions that I may have for possible changes in your project via Canvas.

In [ ]:
# Nice things to have this lecture
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Linear Interpolation

The simplest interpolation we can imagine is doing _connect-the-dots_, and simply
connecting our data samples with straight lines.

![example of linear interpolation](https://zingale.github.io/computational_astrophysics/_images/linear-interpolation.png)

As we see, this only gets the correct values at the end points (for a general function), but it has the nice property that it does not introduce any new minima or maxima.

Given some N points sampling a function, $(x_i, f_i)$, the interpolant connecting the points $i$ and $i+1$ is:

$$f(x) = \frac{f_{i+1} - f_i}{x_{i+1} - x_i} (x - x_i) + f_i$$

for $x_i \le x \le x_{i+1}$

For N points, we have $N-1$ interpolants.

## Piecewise interpolation

This is an example of _piecewise interpolation_.  We don't try to fit all of the
points with a single polynomial, but instead fit a subset of points with a low-order polynomial (in this case, a line).

**Example:** Let's write a function to tabulate $f(x) = x \sin(x)$ on $[0, 5]$ and then use linear interpolation to get the function values. How many function samples do we need to get a 1% error?

In [ ]:
def f(x):
    return x * np.sin(x)

def sample_function(func, npts, *, xmin=0.0, xmax=10.0):
    xx = np.linspace(xmin, xmax, npts)
    return xx, func(xx)

Let's look at this function and 10 samples

In [ ]:
fig, ax = plt.subplots()
xmin = 0.0
xmax = 10.0
N = 10

x_fine = np.linspace(xmin, xmax, 1000)
x_sample, f_sample = sample_function(f, N, xmin=xmin, xmax=xmax)

ax.plot(x_fine, f(x_fine), label="true function")
ax.plot(x_sample, f_sample, marker="o", ls=":", label="linear interpolation")
ax.legend()
plt.show()

Now let's define our interpolate function.  This will take the location and values where the function is sampled, (`xv`, `fv`), and the location where we want to approximate the value of the underlying function, `x0`.

In [ ]:
def interpolate(x0, xv, fv):
    
    # find first x[i] > x0
    idx = np.argwhere(xv > x0)[0][0]
    
    # we want to use this point and the one to the left
    # we'll shift idx to point to the left point
    idx = max(0, idx-1)
    
    slope = (fv[idx+1] - fv[idx]) / (xv[idx+1] - xv[idx])
    return slope * (x0 - xv[idx]) + fv[idx]

Let's just look at $x_0 = 4.5$ (this is near a minimum) and try different number of points and measure the error

In [ ]:
x0 = 4.5

for n in [5, 10, 20, 40, 80, 160]:
    xv, fv = sample_function(f, n)

    finterp = interpolate(x0, xv, fv)
    fexact = f(x0)
    err = np.abs(finterp - fexact)
    print(f"{n:4} points, error = {err}")

In this case, after 80 points, our error is < 1%.  

We also see that we appear to converge slightly better than 2nd order

# Lagrange Interpolation

Imagine that we have a dataset with $N$ points, $\{x_0, x_1, \ldots, x_{N-1}\}$ and associated function values
$\{f_0, f_1, \ldots, f_{N-1}\}$.  We can write a general polynomial interpolation routine that passes through _all $N$ points_ using the [Lagrange polynomial](https://en.wikipedia.org/wiki/Lagrange_polynomial).

The basic idea is to create $N$ _basis functions_ of the form:

$$\ell_m(x) = \prod_{n=0, n\ne m}^{N-1} \frac{x - x_n}{x_m - x_n}$$

This has the property that:

$$\ell_m(x_n) = \delta_{mn}$$

We can then write our interpolant as:

$$f(x) = \sum_{m=0}^{N-1} \ell_m f_m$$

## Quadratic example

Consider fitting a quadratic to three points:

$$(x_0, f_0), (x_1,f_1), (x_2, f_2)$$

Let's imagine that they are evenly spaced, with a spacing $\Delta x$.

The three basis functions are:

\begin{align*}
\ell_0 &= \frac{x - x_1}{x_0 - x_1} \cdot
          \frac{x-x_2}{x_0 - x_2} =
         \frac{(x-x_1)(x-x_2)}{2 \Delta x^2} \\
\ell_1 &= \frac{x - x_0}{x_1 - x_0} \cdot
          \frac{x-x_2}{x_1 - x_2} =
         -\frac{(x-x_0)(x-x_2)}{\Delta x^2} \\
\ell_2 &= \frac{x - x_0}{x_2 - x_0} \cdot
          \frac{x-x_1}{x_2 - x_1} =
         \frac{(x-x_0)(x-x_1)}{2 \Delta x^2}
\end{align*}

and the quadratic polynomial is:

$$
f(x) = \frac{(x-x_1)(x-x_2)}{2 \Delta x^2} f_0 -
          \frac{(x-x_0)(x-x_2)}{\Delta x^2} f_1 +
          \frac{(x-x_0)(x-x_1)}{2 \Delta x^2} f_2
$$

Let's implement this

In [ ]:
def f_lagrange(x0, xv, fv):
    """ xv, fv are the 3 points, x9 is where we want to evaluate """
    dx = x[1] - x[0]

    return (x0 - xv[1])*(x0-xv[2])*fv[0]/(2.0*dx**2) - \
           (x0 - xv[0])*(x0-xv[2])*fv[1]/(dx**2) + \
           (x0 - xv[0])*(x0-xv[1])*fv[2]/(2.0*dx**2)

We'll look at several different sets of points.  For each, we'll plot the interpolated parabola through the 3 points.

In [ ]:
fig, ax = plt.subplots()

# grid points where our data is
x = np.array([0.0, 0.5, 1.0])

f1 = np.array([1.0, -1.0, 1.0/3.0])
f2 = np.array([0.5, 0.5, 0.1])
f3 = np.array([-0.3, 1.3, 1.0])

xfine = np.linspace(0.0, 1.0, 200)

for i,farray in enumerate((f1, f2, f3)):
    ax.scatter(x, farray, marker="x", s=25)
    ax.plot(xfine, f_lagrange(xfine, x, farray), label=f"f{i+1}")
ax.legend()
plt.show()

Notice that the interpolated values exceed the range of $\{f_1, f_2, f_3\}$.  For some applications we may not to introduce new extrema.

## Runge's phenomena

Lagrange interpolation also suffers from [Runge's phenomenon](https://en.wikipedia.org/wiki/Runge%27s_phenomenon) if used with equally spaced points.  This gives rise to larges osciallations at the end of the interpolating interval if we use very high-order polynomial.

Let's write a general Lagrange interpolation class and test it out on some different functions.

In [ ]:
class LagrangePoly:
    """ a general class for creating a Lagrange polynomial 
    representation of a function """

    def __init__(self, xmin, xmax, N, func, chebyshev=False):
        self.N = N
        self.xmin = xmin
        self.xmax = xmax
        self.func = func

        if chebyshev:
            k = np.arange(N) + 1
            self.xp = 0.5 * (xmin + xmax) + 0.5 * (xmax - xmin) * np.cos((2 * k -1) * np.pi / (2 * N))
        else:
            self.xp = np.linspace(self.xmin, self.xmax, self.N)
        self.fp = func(self.xp)

    def evalf(self, x0):
        """ given a point x0 and a function func, fit a Lagrange
        polynomial through the control points and return the
        interpolated value at x """

        f = 0.0

        # sum over points
        for m in range(len(self.xp)):

            # create the Lagrange basis polynomial for point m
            l = 1

            for n in range(len(self.xp)):
                if n == m:
                    continue

                l *= (x0 - self.xp[n])/(self.xp[m] - self.xp[n])

            f += self.fp[m]*l

        return f

Let's test it out on a $\tanh$ profile

In [ ]:
def func(x):
    return 0.5 * (1.0 + np.tanh((x - 1.0) / 0.1))

In [ ]:
N = 11
xmin = 0.0
xmax = 2.0

l = LagrangePoly(xmin, xmax, N, func)

Let's compute the interpolated value at a lot of points and also compute the error with respect to the exact function value.

In [ ]:
# xx are the finely grided data that we will interpolate at to get
# the interpolated function values ff
xx = np.linspace(xmin, xmax, 200)
ff = np.zeros_like(xx)

for n, v in enumerate(xx):
    ff[n] = l.evalf(v)

# exact function values at the interpolated points
fexact = func(xx)

In [ ]:
def plot_result(xx, ff, fexact):

    # compute the error
    e = fexact - ff

    fig = plt.figure()
    ax = fig.add_subplot(211)

    ax.scatter(l.xp, l.fp, marker="x", s=30)
    ax.plot(xx, ff)

    ax.plot(xx, fexact, color="0.5")

    ax = fig.add_subplot(212)

    ax.plot(xx, e)
    ax.set_xlabel("x")
    ax.set_ylabel("error")
    fig.tight_layout()
    plt.show()

In [ ]:
plot_result(xx, ff, fexact)

**Question:** What happens as we increase the number of data samples, $N$?

## Chebyshev nodes

If we are free to pick the points where we sample the function, we can do a lot better, and minimize the Runge's phenomena.  Sampling $f(x)$ at the [Chebyshev nodes](https://en.wikipedia.org/wiki/Chebyshev_nodes):

$$x_k = \frac{1}{2}(a +b) + \frac{1}{2} (b-a) \cos \left (\frac{2k - 1}{2n}\pi \right )\, , \quad k = 1, \ldots n$$

where $[a, b]$ is the domain, works well.

Let's try this out

In [ ]:
l = LagrangePoly(xmin, xmax, N, func, chebyshev=True)

for n, v in enumerate(xx):
    ff[n] = l.evalf(v)

# exact function values at the interpolated points
fexact = func(xx)

plot_result(xx, ff, fexact)

# Exercises (10 pt)

## Exercise 1 - Muddiest Moment (2 pt)

What was the most confusing part of this lecture?

**Answer here**


## Exercise 2 - Interpolating Reaction Rates (8 pt)

[Supernovae shine](https://en.wikipedia.org/wiki/Supernova#Light_curves) because of the radioactive decay of ${}^{56}\mathrm{Ni}$, which is one of the end-points
of fusion reactions of light elements.  At high densities, once this ${}^{56}\mathrm{Ni}$ is made, the electron-capture reaction:

$${}^{56}\mathrm{Ni} + e^- \rightarrow {}^{56}\mathrm{Co} + \nu_e$$

is important in determining how much ${}^{56}\mathrm{Ni}$ remains compared to other stable iron-group nuclei.
A similar transformation can happen via $\beta^+$ decay:

$${}^{56}\mathrm{Ni} \rightarrow {}^{56}\mathrm{Co} + e^+ + \nu_e$$

We call these reactions *weak nuclear reactions*, because they involve changing a neutron to a proton, and thus
the weak nuclear force.

These reaction rates are temperature and density dependent (in terms of electron density, $\rho Y_e$, where $Y_e$ is the electron fraction)
and evaluations of them are usually provided
as tables requiring reaction networks to interpolate.

Here we'll learn how to work with a tabulated reaction rate.

### Exercise 2(a) - Accessing the data (2 pt)

We will download a table from [Langanke & Martínez-Pinedo 2001](https://www.sciencedirect.com/science/article/abs/pii/S0092640X01908654).  This has been cleaned up a little
from the tables provided there, combining both the e-capture and $\beta^+$-decay into a single rate.

To download, type the command
```bash
curl -O https://zingale.github.io/computational_astrophysics/_sources/interpolation/application-interpolating-reaction-rates.ipynb
```
into your terminal command line.

Verify the download by typing
```python
!head 56ni-56co_electroncapture.dat
```
in the cell below

In [ ]:
## Answer here


### Exercise 2(b) - Sorting and extracting the data (3 pt)

We see that the data is in columns with most quantities stored as logs.  We're interest in the first column, $\log(\rho Y_e)$, the second column, $\log(T)$, and the sixth column, $\log(\lambda)$ (the reaction rate). The $\log()$ are base-10 logarithms.

Use `np.genfromtxt()` to read this, which can interpret a comment line as labels for the data, giving us an
easy way to index the columns, with the following command:
```python
data = np.genfromtxt("56ni-56co_electroncapture.dat", skip_header=3, names=True)
```

From the table, we can learn that there are 13 temperatures and 11 densities in the tabulation:
```python
ntemp = 13
nrho = 11
```

First, extract the unique densities ($\log(\rho Y_e)$) from the `LogrhoY` column, by taking every `ntemp` value:
```python
logrho = data["LogrhoY"][::ntemp]
logrho
```

Next, examine the unique temperatures ($\log(T)$) from the `Logtemp` column:
```python
logT = data["Logtemp"][0:ntemp]
logT
```

Note that temperatures are not evenly spaced.  We need to make sure that we take that
into account when we do the interpolation.

Finally, we'll access the rate, $\lambda$, from the `Logecaprate` column, and reshape it into a two-dimensional table:
```python
lograte = data["Logecaprate"].reshape(nrho, ntemp)
```
print some of the values using
```python
lograte[0:4, 0:4]
```

### Exercise 2(c) - Creating the bilinear interpolator (4 pt)

We'll do [bilinear interpolation](https://en.wikipedia.org/wiki/Bilinear_interpolation), using taking the 4 points in the table that surround the $(\rho Y_e, T)$ at which we wish to evaluate
the rate and fit it with a function:

\begin{align*}
\log [\lambda(\rho Y_e, T)] = {}&a (\log(\rho Y_e) - \log(\rho Y_e)_i)(\log(T) - \log(T)_j) + \\
                       &b (\log(\rho Y_e) - \log(\rho Y_e)_i) + \\
                       &c (\log(T) - \log(T)_j) + \\
                       &d \\
                    \equiv {}&a \cdot \delta \log (\rho Y_e) \cdot \delta \log(T) + \\
                       &b \cdot \delta \log(\rho Y_e) + \\
                       &c \cdot \delta \log(T) + \\
                       &d
\end{align*}

here we take $(i, j)$ as the reference point for the interpolant, and we have defined the quantity
\begin{equation}
\delta \log(X) \equiv \log(X) - \log(X_i).
\end{equation}

We need to find the 4 unknowns, $\{a, b, c, d\}$, using the information from the table.

Visually, this appears as:

![bilinear interpolation](https://zingale.github.io/computational_astrophysics/_images/bilinear_ecapture.png)

where the $\times$ is where we want to know the rate.

We can do this algebraically, by evaluating our interpolant at our 4 points:

\begin{align*}
\log(\lambda_{i,j}) &= d \\
\log(\lambda_{i+1,j}) &= b \Delta \log (\rho Y_e) + d \\
\log(\lambda_{i,j+1}) &= c \Delta \log (T) + d \\
\log(\lambda_{i+1,j+1}) &= a \Delta \log (\rho Y_e) \Delta \log (T) + b \Delta \log (\rho Y_e) + c \Delta \log (T) + d
\end{align*}

where we have defined the quantity
\begin{equation}
\Delta \log(X) \equiv \log(X_{i+1}) - \log(X_i).
\end{equation}
for the changes in quantities across the entire grid.

Then we see:

$$d = \log(\lambda_{i,j})$$

$$b = \frac{\log(\lambda_{i+1,j}) - \log(\lambda_{i,j})}{\Delta \log (\rho Y_e)}$$

$$c = \frac{\log(\lambda_{i,j+1}) - \log(\lambda_{i,j})}{\Delta \log (T)}$$

$$a = \frac{\log(\lambda_{i+1,j+1}) - \log(\lambda_{i+1,j}) - \log(\lambda_{i,j+1}) + \log(\lambda_{i,j})}{\Delta\log (\rho Y_e) \Delta \log (T)}$$

Lets implement an interplative function to do just this. Filling in the values indicated below, finish the code for the bi-linear interplator:

In [ ]:
def interp(logrhoY_v, logT_v, lograte_t, logrhoY0, logT0):
    """interpolate lograte_t to find the rate at a point (logrhoY0, logT0).
    Here logrhoY_v and logT_v are the points where the rate is tabulated
    Also, you will call e.g. log(lambda_i,j) via lograte_t[i, j]"""

    i = np.argwhere(logrhoY_v > logrhoY0)[0][0] - 1
    j = np.argwhere(logT_v > logT0)[0][0] - 1

    # Values for changes in log(rho Y_e) and log(T) across entire grid
    DlogrhoY = logrhoY_v[i+1] - logrhoY_v[i]
    DlogT = logT_v[j+1] - logT_v[j]

    # Input values for the coefficients here
    a = 
    b = 
    c = 
    d = 

    # Values for the changes up to the interpolated point:
    dlogrhoY = logrhoY0 - logrhoY_v[i]
    dlogT = logT0 - logT_v[j]

    # Return the interpolated log rate here
    return 

### Exercise 2(d) - Evaluating the bilinear interpolator (2 pt)

Evaluate your bilinear interpolator for $\log(\rho Y_e) = 3.5$ and $\log(T) = 9.4$. Print a f-string with the (non-log) value of $\rho Y_e$ and $T$, and verify your reaction rate result, which is supposed to be about $\lambda = 1.6 \times 10^{-5}$.